```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b done;
    class A2 current;
    class A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 02 — Corpus Analysis & Understanding: Exploring the Philosophy Corpus

This notebook is the first “research-facing” step of the course workflow: we move from *having a corpus* to *understanding what it contains* and whether it is fit for answering our questions about the spread of knowledge over time. We perform core quality checks (encoding, missingness, duplicates, language consistency), make explicit preprocessing choices (normalization and segmentation), and produce an initial descriptive portrait of the corpus (document lengths, basic frequency summaries, and coverage of key metadata fields).

**Outputs from this notebook** are saved to disk and reused in later sessions, so downstream analyses (time trends, association, representations, and modeling) can build on a stable, documented corpus rather than re-running ingestion and cleaning from scratch.



## Setup

In [ ]:
!pip install nltk ipywidgets

In [ ]:
# -----------------------------
# Import
# -----------------------------

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Optional
import numpy as np
import re

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("✓ Libraries loaded")

# -----------------------------
# Paths
# -----------------------------
CORPUS_DIR = Path('./data')
TEXTS_RAW_DIR = CORPUS_DIR / 'raw'
TEXTS_CLEANED_DIR = CORPUS_DIR / 'processed' / 'cleaned'
METADATA_FILE = Path('./analysis/tables/nb01-selected-metadata.csv')
MANUAL = Path('./analysis/tables/nb01-manual-enrichments.csv')
DOWNLOAD_LOG = Path('./analysis/reports/nb01-download_log.json')
OUTPUT_DIR = Path('./analysis')

print(f"Corpus directory: {CORPUS_DIR.absolute()}")
print(f"Raw texts: {TEXTS_RAW_DIR.absolute()}")
print(f"Cleaned texts: {TEXTS_CLEANED_DIR.absolute()}")
print(f"Metadata: {METADATA_FILE.absolute()}")

# -----------------------------
# Parameters
# -----------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

## Part 1: Load Corpus and Metadata

In [ ]:
meta = pd.read_csv(METADATA_FILE)

print(f"✓ Loaded metadata for {len(meta)} books")
print(f"\nColumns available:")
for col in meta.columns:
    print(f"  - {col}")

manual_meta = pd.read_csv(MANUAL)
print(f"\n✓ Loaded manual enrichments for {len(manual_meta)} books")
print(f"\nColumns available:")
for col in manual_meta.columns:
    print(f"  - {col}")

In [ ]:
def merge_manual_enrichments(
    metadata_path: Path,
    manual_path: Path,
    key_col: str = "pg_id",
    enrich_cols: List[str] = ["wikipedia_url", "author_deathdate", "publication_year"],
    output_path: Optional[Path] = None,
) -> pd.DataFrame:
    """
    Merge manual enrichments into the metadata file.

    Parameters
    ----------
    metadata_path : Path
        Path to the main metadata CSV (must contain `key_col`).
    manual_path : Path
        Path to the manual enrichment CSV (contains `key_col` and the enrichment columns).
    key_col : str, default "Text#"
        Column used as the join key (book ID).
    enrich_cols : list of str, default ["wikipedia_url", "author_deathdate", "publication_year"]
        Columns to merge from the manual file.
    output_path : Path, optional
        If provided, the merged DataFrame is saved to this path.

    Returns
    -------
    pd.DataFrame
        The merged DataFrame with updated enrichment columns.
    """
    # Read the files
    metadata = pd.read_csv(metadata_path)
    manual = pd.read_csv(manual_path)

    # Ensure key column is string for consistent merging
    metadata[key_col] = metadata[key_col].astype(str)
    manual[key_col] = manual[key_col].astype(str)

    # Keep only the relevant columns from manual (key + enrich_cols)
    manual_subset = manual[[key_col] + enrich_cols].copy()

    # Merge: left join to keep all metadata rows
    merged = metadata.merge(manual_subset, on=key_col, how='left', suffixes=('', '_manual'))

    # For each enrichment column, update the metadata column with manual values
    # Only overwrite if the metadata value is missing (NaN)
    for col in enrich_cols:
        manual_col = f"{col}_manual"
        if manual_col in merged.columns:
            # Fill missing values in the original column with manual values
            merged[col] = merged[col].fillna(merged[manual_col])
            # Drop the temporary manual column
            merged.drop(columns=[manual_col], inplace=True)

    # Print summary: count of missing values per enrichment column after merge
    print("\n" + "=" * 60)
    print("MERGE SUMMARY")
    print("=" * 60)
    print(f"Total rows in metadata: {len(metadata)}")
    print(f"Total rows in manual: {len(manual)}")
    print("\nMissing values after merge (per column):")
    for col in enrich_cols:
        missing_count = merged[col].isna().sum()
        missing_pct = (missing_count / len(merged)) * 100
        print(f"  {col}: {missing_count} missing ({missing_pct:.1f}%)")

    # Show example of rows where enrichments are still missing (first 5)
    print("\nSample of rows still missing enrichment data:")
    missing_mask = merged[enrich_cols].isna().any(axis=1)
    sample_missing = merged[missing_mask].head(5)
    if len(sample_missing) > 0:
        display(sample_missing[[key_col, 'title'] + enrich_cols])
    else:
        print("\nAll rows have complete enrichment data!")

    # Save if output path provided
    if output_path:
        merged.to_csv(output_path, index=False)
        print(f"\nMerged file saved to: {output_path}")

    return merged

In [ ]:
# Define paths
METADATA_FILE = Path('./analysis/tables/nb01-selected-metadata.csv')
MANUAL_FILE = Path('./analysis/tables/nb01-manual-enrichments.csv')
OUTPUT_FILE = Path('./analysis/tables/nb02-metadata.csv')

# Fill the gap

In [ ]:
# Complete the code below to apply the merging function correctly
# =============================================== YOUR CODE HERE ===============================================
df_enriched = merge_manual_enrichments(
    metadata_path = ,       # Add the variable to which we assigned the path to the metadata
    manual_path = ,         # Add the variable name to which we assigned the path to the enriched metadata
    key_col=,               # Which is the key reference column on which to merge?
    enrich_cols=[],         # Which are the three columns that should be enriched
    output_path=OUTPUT_FILE
)

# Fill the gap

In [ ]:
# =============================================== YOUR CODE HERE ===============================================
df_metadata = # Load the CSV that was stored in nb02-metadata.csv and assign it to df_metadata with Pandas

print(f"✓ Loaded updated metadata for {len(df_metadata)} books")
print(f"\nColumns available:")
for col in df_metadata.columns:
    print(f"  - {col}")

In [ ]:
# Check if any publication_year values are missing
missing_mask = df_metadata["publication_year"].isna()

if missing_mask.any():
    # Store IDs of rows missing publication_year
    missing_ids = df_metadata.loc[missing_mask, "pg_id"].tolist()
    print(f"Found {len(missing_ids)} rows with missing publication_year.")
    print(f"IDs: {missing_ids[:10]}{'...' if len(missing_ids) > 10 else ''}")

    # First fallback: fill with author_deathdate
    df_metadata["publication_year"] = (
        pd.to_numeric(df_metadata["publication_year"], errors="coerce")
        .fillna(pd.to_numeric(df_metadata["author_deathdate"], errors="coerce"))
        .astype("Int64")
    )
    filled_with_death = len(missing_ids) - df_metadata["publication_year"].isna().sum()
    print(f"   → Filled {filled_with_death} rows using author_deathdate.\n")

    # Check if any still missing
    still_missing_mask = df_metadata["publication_year"].isna()
    if still_missing_mask.any():
        # Store IDs of rows still missing
        still_missing_ids = df_metadata.loc[still_missing_mask, "pg_id"].tolist()
        print(f"   ⚠️  {len(still_missing_ids)} rows still missing → trying author_birthdate.")

        # Second fallback: fill remaining with author_birthdate
        df_metadata["publication_year"] = (
            pd.to_numeric(df_metadata["publication_year"], errors="coerce")
            .fillna(pd.to_numeric(df_metadata["author_birthdate"], errors="coerce"))
            .astype("Int64")
        )
        filled_with_birth = len(still_missing_ids) - df_metadata["publication_year"].isna().sum()
        print(f"   → Filled {filled_with_birth} additional rows using author_birthdate.\n")

        # Final check
        final_missing = df_metadata["publication_year"].isna().sum()
        if final_missing > 0:
            final_missing_ids = df_metadata.loc[df_metadata["publication_year"].isna(), "pg_id"].tolist()
            print(f"      ⚠️  {final_missing} rows still missing publication_year.")
            print(f"      IDs: {final_missing_ids[:10]}{'...' if len(final_missing_ids) > 10 else ''}")
        else:
            print("All publication_year values now filled!")
    else:
        print("All publication_year values now filled!")
else:
    print("No missing publication_year values.")

In [ ]:
# Count downloaded texts
raw_files = list(TEXTS_RAW_DIR.glob('pg*.txt'))
cleaned_files = list(TEXTS_CLEANED_DIR.glob('pg*.txt'))

print(f"\nDownloaded texts:")
print(f"  Raw files: {len(raw_files)}")
print(f"  Cleaned files: {len(cleaned_files)}")
print(f"  Expected (from metadata): {len(df_metadata)}")

# Check if counts match
if len(cleaned_files) < len(df_metadata):
    print(f"\n⚠️  Missing {len(df_metadata) - len(cleaned_files)} texts")
elif len(cleaned_files) == len(df_metadata):
    print(f"\n✓ All expected texts downloaded successfully!")

In [ ]:
# Load download log if available
if DOWNLOAD_LOG.exists():
    with open(DOWNLOAD_LOG, 'r') as f:
        download_log = json.load(f)
    
    # Access nested structure
    download_data = download_log.get('download', download_log)  # Fallback to root if no 'download' key
    
    print(f"\nDownload log:")
    print(f"  Successful: {len(download_data.get('successful', []))}")
    print(f"  Failed: {len(download_data.get('failed', []))}")
    
    if download_data.get('failed'):
        print(f"\n  Failed book IDs: {download_data['failed']}")
else:
    print("\nNo download log found")
    download_log = None

# Part 2: Raw vs Cleaned Text Comparison

Compare original downloaded texts with cleaned versions to see what was removed.

In [ ]:
# Select 5 random texts for comparison
import random

random.seed(42)  # For reproducibility
sample_files = random.sample(cleaned_files, min(5, len(cleaned_files)))

print("Sample texts for raw vs cleaned comparison:")
print("=" * 80)

for cleaned_file in sample_files:
    book_id = cleaned_file.stem.replace('pg', '')
    raw_file = TEXTS_RAW_DIR / cleaned_file.name
    
    # Get title from metadata
    book_info = df_metadata[df_metadata['pg_id'] == int(book_id)]
    title = book_info['title'].values[0] if len(book_info) > 0 else "Unknown"
    
    # Load texts
    with open(raw_file, 'r', encoding='utf-8') as f:
        raw_text = f.read()
    
    with open(cleaned_file, 'r', encoding='utf-8') as f:
        cleaned_text = f.read()
    
    print(f"\nBook ID {book_id}: {title[:60]}...")
    print(f"  Raw size: {len(raw_text):,} characters")
    print(f"  Cleaned size: {len(cleaned_text):,} characters")
    print(f"  Removed: {len(raw_text) - len(cleaned_text):,} characters ({(1 - len(cleaned_text)/len(raw_text))*100:.1f}%)")
    
    # Show first 200 chars of raw (header)
    print(f"\n  Raw text start:")
    print(f"  {raw_text[:200]}...")
    
    # Show first 200 chars of cleaned
    print(f"\n  Cleaned text start:")
    print(f"  {cleaned_text[:200]}...")
    print("-" * 80)

In [ ]:
from IPython.display import HTML, display

def display_texts_side_by_side(book_id, max_chars=1000):
    """Display raw and cleaned texts side by side."""
    raw_file = TEXTS_RAW_DIR / f'pg{book_id}.txt'
    cleaned_file = TEXTS_CLEANED_DIR / f'pg{book_id}.txt'
    
    with open(raw_file, 'r', encoding='utf-8') as f:
        raw_text = f.read()[:max_chars]
    
    with open(cleaned_file, 'r', encoding='utf-8') as f:
        cleaned_text = f.read()[:max_chars]
    
    # Get title from metadata
    book_info = df_metadata[df_metadata['pg_id'] == int(book_id)]
    title = book_info['title'].values[0] if len(book_info) > 0 else "Unknown"
    
    html = f"""
    <div style="border: 1px solid #ccc; padding: 10px; margin: 20px 0;">
        <h3>Book ID {book_id}: {title[:80]}</h3>
        <div style="display: flex; gap: 20px;">
            <div style="flex: 1; border-right: 2px solid #999; padding-right: 10px;">
                <h4 style="color: #d9534f;">RAW (first {max_chars} chars)</h4>
                <pre style="white-space: pre-wrap; font-size: 11px; max-height: 400px; overflow-y: auto;">{raw_text}</pre>
            </div>
            <div style="flex: 1; padding-left: 10px;">
                <h4 style="color: #5cb85c;">CLEANED (first {max_chars} chars)</h4>
                <pre style="white-space: pre-wrap; font-size: 11px; max-height: 400px; overflow-y: auto;">{cleaned_text}</pre>
            </div>
        </div>
        <div style="margin-top: 10px; font-style: italic; color: #666;">
            Raw: {len(open(raw_file).read()):,} chars | 
            Cleaned: {len(open(cleaned_file).read()):,} chars | 
            Removed: {len(open(raw_file).read()) - len(open(cleaned_file).read()):,} chars
        </div>
    </div>
    """
    display(HTML(html))

In [ ]:
# Display 5 random texts
sample_ids = random.sample(cleaned_files[:], min(5, len(cleaned_files)))
for file in sample_ids:
    book_id = file.stem.replace('pg', '')
    display_texts_side_by_side(book_id, max_chars=1000)

# Fill the gap
1. Run the next cell and generate the csv file in which you will write your observations.
2. Select 5 texts to analyse randomly
3. Compare raw vs cleaned versions
4. Document observations in the CSV
5. Think adversarially: which other texts could display the same issues?
6. Select 5 more texts specifically trying to catch more of the issues you have found.
7. Document observations in the CSV

In [ ]:
# Generate CSV template for student text comparison analysis
text_quality_check_0 = []

for i in range(1, 6):
    text_quality_check_0.append({
        'student_name': 'STUDENT_NAME', # Overwrite with your name
        'text_nid': i,
        'title': '',
        'raw_issues_observed': '',  # e.g., "Headers present, encoding issues"
        'cleaning_effective': '',   # yes/no/partial/unsure
        'remaining_problems': '',   # Any issues still in cleaned version
        'notes': ''
    })

quality_0 = pd.DataFrame(text_quality_check_0)
quality_0_file = OUTPUT_DIR / 'tables' / f'nb02-text_quality_check_0_{text_quality_check_0[0]['student_name']}.csv'
quality_0.to_csv(quality_0_file, index=False)

print(f"Student template saved: {quality_0_file}")

## Part 3: Text-Level Statistics

In [ ]:
# Calculate statistics for all cleaned texts
print("Calculating text-level statistics...")

text_stats = []

for text_file in cleaned_files:
    book_id = int(text_file.stem.replace('pg', ''))
    
    with open(text_file, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Basic counts
    char_count = len(text)
    words = text.split()
    word_count = len(words)
    
    # Vocabulary richness (Type-Token Ratio)
    unique_words = len(set([w.lower() for w in words]))
    ttr = unique_words / word_count if word_count > 0 else 0
    
    text_stats.append({
        'pg_id': book_id,
        'filename': text_file.name,
        'char_count': char_count,
        'word_count': word_count,
        'unique_words': unique_words,
        'type_token_ratio': ttr
    })

df_stats = pd.DataFrame(text_stats)

print(f"✓ Calculated statistics for {len(df_stats)} texts")

In [ ]:
# Display summary statistics
print("\n" + "=" * 60)
print("TEXT-LEVEL STATISTICS SUMMARY")
print("=" * 60)

print(f"\nWord counts:")
print(f"  Total: {df_stats['word_count'].sum():,}")
print(f"  Mean: {df_stats['word_count'].mean():,.0f}")
print(f"  Median: {df_stats['word_count'].median():,.0f}")
print(f"  Min: {df_stats['word_count'].min():,}")
print(f"  Max: {df_stats['word_count'].max():,}")

print(f"\nVocabulary richness (Type-Token Ratio):")
print(f"  Mean: {df_stats['type_token_ratio'].mean():.3f}")
print(f"  Median: {df_stats['type_token_ratio'].median():.3f}")

print(f"\nTotal unique words across corpus: {df_stats['unique_words'].sum():,}")
print(f"Corpus size: {df_stats['char_count'].sum() / (1024 * 1024):.2f} MB")

In [ ]:
# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax1 = axes[0]
ax1.hist(df_stats['word_count'], bins=50, color='teal', edgecolor='black', alpha=0.7)
ax1.axvline(df_stats['word_count'].median(), color='crimson', linestyle='--', 
            label=f'Median: {df_stats["word_count"].median():,.0f}')
ax1.axvline(df_stats['word_count'].mean(), color='orange', linestyle='--', 
            label=f'Mean: {df_stats["word_count"].mean():,.0f}')
ax1.set_xlabel('Word Count')
ax1.set_ylabel('Number of Texts')
ax1.set_title('Distribution of Text Lengths')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot
ax2 = axes[1]
ax2.boxplot(df_stats['word_count'], orientation='vertical')
ax2.set_ylabel('Word Count')
ax2.set_title('Text Length Distribution (Box Plot)')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb02-text_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: text_length_distribution.png")

In [ ]:
# Vocabulary richness vs text length
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

ax.scatter(
    df_stats['word_count'], 
    df_stats['type_token_ratio'], 
    alpha=0.5, 
    s=50,
    c='teal'
)
ax.set_xlabel('text length (words)')
ax.set_ylabel('type-token ratio (vocabulary richness)')
ax.set_title('Vocabulary richness vs text length')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb02-vocabulary_richness.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: nb02-vocabulary_richness.png")
print("Note: Type-Token Ratio typically decreases with text length")
print("(longer texts reuse vocabulary more)")

# Part 4: Corpus-Level Analysis

In [ ]:
# Merge text statistics with metadata
df_complete = df_metadata.merge(df_stats, left_on='pg_id', right_on='pg_id', how='left')

print(f"✓ Merged metadata with text statistics")
print(f"  Total records: {len(df_complete)}")
print(f"  Records with text stats: {df_complete['word_count'].notna().sum()}")

# Handle missing texts
missing_texts = df_complete[df_complete['word_count'].isna()]
if len(missing_texts) > 0:
    print(f"\n⚠️  {len(missing_texts)} texts have no statistics (not downloaded or failed)")
    print(f"  Missing IDs: {missing_texts['pg_id'].tolist()}")

### 4.1 Temporal Distribution

In [ ]:
# Temporal analysis using publication year
df_dated = df_complete[df_complete['publication_year'].notna()].copy()
df_dated['publication_year'] = pd.to_numeric(df_dated['publication_year'], errors='coerce')
df_dated = df_dated[df_dated['publication_year'].notna()]  # keep only rows with valid year
df_dated['pub_century'] = (df_dated['publication_year'] // 100) + 1

print(f"\nTEMPORAL ANALYSIS USING PUBLICATION YEAR")
print(f"Texts with publication year: {len(df_dated)} / {len(df_complete)}")
print(f"Date range: {df_dated['publication_year'].min():.0f} - {df_dated['publication_year'].max():.0f}")

In [ ]:
# Temporal visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Texts per century
ax1 = axes[0]
century_counts = df_dated['pub_century'].value_counts().sort_index()
century_labels = [f"{int((c-1)*100)}s" if c > 0 else f"{int((c-1)*100)}s BCE" for c in century_counts.index]

ax1.bar(range(len(century_counts)), century_counts.values, color='teal')
ax1.set_xticks(range(len(century_counts)))
ax1.set_xticklabels(century_labels, rotation=45)
ax1.set_xlabel('Century (publication year)')
ax1.set_ylabel('Number of Texts')
ax1.set_title('Temporal distribution of corpus (by publication year)')
ax1.grid(True, alpha=0.3, axis='y')

# Right: Word count per century
ax2 = axes[1]
century_words = df_dated.groupby('pub_century')['word_count'].sum()
century_words_labels = [f"{int((c-1)*100)}s" if c > 0 else f"{int((c-1)*100)}s BCE" for c in century_words.index]

ax2.bar(range(len(century_words)), century_words.values, color='gold')
ax2.set_xticks(range(len(century_words)))
ax2.set_xticklabels(century_words_labels, rotation=45)
ax2.set_xlabel('Century (publication year)')
ax2.set_ylabel('Total Word Count')
ax2.set_title('Word count distribution by century')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb02-temporal_distribution_publication.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: nb02-temporal_distribution_publication.png")

# Critical question:

_What do these visualisations tell us?_

 =============================================== YOUR THOUGHTS HERE ===============================================



---

In [ ]:
# Texts by period summary
print("\nTexts per century:")
print("=" * 60)
for century, count in century_counts.items():
    century_name = f"{int((century-1)*100)}s" if century > 0 else f"{int((century-1)*100)}s BCE"
    words = century_words.get(century, 0)
    bar = '█' * (count // 5)
    print(f"  {century_name:15} | {count:3} texts | {words:,} words | {bar}")

## 4.2 Author Representation

In [ ]:
# Extract primary author (first in Authors field)
df_complete['primary_author'] = df_complete['authors'].apply(
    lambda x: str(x).split(';')[0].split(',')[0].strip() if pd.notna(x) else 'Unknown'
)

# Count texts per author
author_counts = df_complete['primary_author'].value_counts()

print("\nMost represented authors (by text count):")
print("=" * 60)
for author, count in author_counts.head(20).items():
    print(f"  {author:40} : {count:2} texts")

In [ ]:
# Word count per author (top 20)
author_words = df_complete.groupby('primary_author')['word_count'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

top_authors = author_words.head(20)
ax.barh(range(len(top_authors)), top_authors.values, color='teal')
ax.set_yticks(range(len(top_authors)))
ax.set_yticklabels(top_authors.index)
ax.set_xlabel('Total word count')
ax.set_title('Top 20 authors by word count in corpus')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb02-author_representation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: nb02-author_representation.png")

### 4.3 Subject Distribution

In [ ]:
# Extract all subjects
all_subjects = []
for subjects in df_complete['subjects'].dropna():
    subject_list = [s.strip() for s in str(subjects).split(';')]
    all_subjects.extend(subject_list)

subject_counts = Counter(all_subjects)

print("\nTop 20 subject tags in corpus:")
print("=" * 60)
for subject, count in subject_counts.most_common(20):
    print(f"  {count:3} | {subject}")

In [ ]:
# Visualize top subjects
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

top_subjects = dict(subject_counts.most_common(15))
ax.barh(range(len(top_subjects)), list(top_subjects.values()), color='gold')
ax.set_yticks(range(len(top_subjects)))
ax.set_yticklabels(list(top_subjects.keys()))
ax.set_xlabel('Number of texts')
ax.set_title('Top 15 Subjects in Corpus')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'nb02-subject_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: nb02-subject_distribution.png")

## Part 5: Data Quality Assessment

In [ ]:
# Identify very short texts (< 10,000 words)
short_texts = df_complete[df_complete['word_count'] < 10000].sort_values('word_count')

print("\nVERY SHORT TEXTS (< 10,000 words):")
if len(short_texts) > 0:
    print(f"Found {len(short_texts)} short texts:\n")
    print("=" * 80)
    for idx, row in short_texts.iterrows():
        print(f"  ID {row['pg_id']:5} | {row['word_count']:5,} words | {row['title'][:50]}")
    print("=" * 80)
    print("\nThese might be: essays, letters, incomplete texts, or misclassified items")
else:
    print("No texts shorter than 10,000 words found.")

In [ ]:
# Identify very long texts (> 300,000 words)
long_texts = df_complete[df_complete['word_count'] > 300000].sort_values('word_count', ascending=False)

print("\nVERY LONG TEXTS (> 300,000 words):")
if len(long_texts) > 0:
    print(f"Found {len(long_texts)} very long texts:\n")
    print("=" * 80)
    for idx, row in long_texts.iterrows():
        print(f"  ID {row['pg_id']:5} | {row['word_count']:7,} words | {row['title'][:50]}")
    print("=" * 80)
    print("\nThese might be: complete works, anthologies, or multi-volume collections")
else:
    print("No texts longer than 300,000 words found.")

In [ ]:
# Missing/failed texts
missing_texts = df_complete[df_complete['word_count'].isna()]

print("\nMISSING OR FAILED TEXTS:")
if len(missing_texts) > 0:
    print(f"Found {len(missing_texts)} texts with no data:\n")
    print("=" * 80)
    for idx, row in missing_texts.iterrows():
        print(f"  ID {row['pg_id']:5} | {row['title'][:60]}")
        print(f"         Author: {row['Authors'][:60]}")
else:
    print("✓ All selected texts downloaded successfully!")

# Fill the gap
1. Run the next cell and generate the csv file in which you will write your observations.
2. Inspect 5 _random_ texts
3. Dcument quality issues
5. Think adversarially: what could make our analysis unreliable?
4. Inspect 5 _selected_ texts based on your observations (choose strategically: very short, very long, suspicious titles, etc.)

In [ ]:
# Generate quality inspection template for students
# 5 random + 5 selected

text_quality_check_1 = []

# Section 1: Random texts
for i in range(1, 6):
    text_quality_check_1.append({
        'student_name': 'STUDENT_NAME', # Overwrite with your name
        'inspection_type': 'random',
        'text_number': i,
        'pg_id': '',
        'title': '',
        'word_count': '',
        'is_philosophy': '',  # Yes/No/Unclear
        'text_quality': '',   # Good/Fair/Poor
        'issues_found': '',   # e.g., "Scanning errors, missing pages, wrong language"
        'recommendations': '' # Keep/Remove/Further review
    })

# Section 2: Selected texts (students choose)
for i in range(1, 6):
    text_quality_check_1.append({
        'student_name': 'STUDENT_NAME', # Overwrite with your name
        'inspection_type': 'selected',
        'text_number': i,
        'pg_id': '',
        'title': '',
        'word_count': '',
        'is_philosophy': '',
        'text_quality': '',
        'issues_found': '',
        'recommendations': ''
    })

quality_1_df = pd.DataFrame(text_quality_check_1)
quality_1_file = OUTPUT_DIR / 'tables' / f'nb02-text_quality_check_1_{text_quality_check_1[0]['student_name']}.csv'
quality_1_df.to_csv(quality_1_file, index=False)

print(f"\n✓ Quality inspection file saved: {quality_1_file}")

# Part 6: Basic Preprocessing Preview

Preview of text preprocessing concepts we'll use in later notebooks.

In [ ]:
# Load a sample text for preprocessing demo
sample_file = cleaned_files[0]
book_id = sample_file.stem.replace('pg', '')

with open(sample_file, 'r', encoding='utf-8') as f:
    sample_text = f.read()[:2000]  # First 2000 characters

# Get title
book_info = df_metadata[df_metadata['pg_id'] == int(book_id)]
title = book_info['title'].values[0] if len(book_info) > 0 else "Unknown"

print(f"Sample text: {title}")
print("=" * 80)
print(sample_text)
print("\n" + "=" * 80)

In [ ]:
# Tokenization - splitting text into small units
from nltk.tokenize import word_tokenize

tokens = word_tokenize(sample_text)

print("\n1. TOKENIZATION - Splitting text into words")
print("=" * 80)
print(f"Original text length: {len(sample_text)} characters")
print(f"Number of tokens: {len(tokens)}")
print(f"\nFirst 50 tokens:")
print(tokens[:50])

In [ ]:
# Lowercasing and punctuation removal
import string

# Lowercase
tokens_lower = [token.lower() for token in tokens]

# Remove punctuation
tokens_no_punct = [token for token in tokens_lower if token not in string.punctuation]

print("\n2. LOWERCASING & PUNCTUATION REMOVAL")
print("=" * 80)
print(f"Original tokens: {len(tokens)}")
print(f"After removing punctuation: {len(tokens_no_punct)}")
print(f"\nFirst 50 cleaned tokens:")
print(tokens_no_punct[:50])

In [ ]:
# Stopword removal
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Remove stopwords
tokens_no_stop = [token for token in tokens_no_punct if token not in stop_words]

print("\n3. STOPWORD REMOVAL")
print("=" * 80)
print(f"Tokens before stopword removal: {len(tokens_no_punct)}")
print(f"Tokens after stopword removal: {len(tokens_no_stop)}")
print(f"Stopwords removed: {len(tokens_no_punct) - len(tokens_no_stop)}")
print(f"\nStopwords (first 20): {list(stop_words)[:20]}")
print(f"\nFirst 50 content words (no stopwords):")
print(tokens_no_stop[:50])

In [ ]:
# Most common content words
word_freq = Counter(tokens_no_stop)

print("\n4. WORD FREQUENCY (after preprocessing)")
print("=" * 80)
print("Most common content words in sample text:")
for word, count in word_freq.most_common(20):
    print(f"  {word:20} : {count:3}")

print("\nThese preprocessing steps will be crucial for:")
print("   - Frequency analysis")
print("   - Topic modeling")
print("   - Word embeddings")
print("   - Text classification")

In [ ]:
from IPython.display import HTML, display

def highlight_stopwords(text, max_words=200):
    """Show a text snippet with stopwords highlighted in red."""
    tokens = word_tokenize(text.lower())[:max_words]
    
    highlighted = []
    for token in tokens:
        if token in stop_words:
            highlighted.append(f'<span style="background-color: #ffcccc; padding: 2px;">{token}</span>')
        else:
            highlighted.append(f'<span style="background-color: #ccffcc; padding: 2px;"><b>{token}</b></span>')
    
    html = f"""
    <div style="line-height: 2; font-size: 14px; padding: 20px; border: 1px solid #ccc;">
        <h4>Stopwords Highlighted (first {max_words} tokens)</h4>
        <p><span style="background-color: #ffcccc; padding: 5px;">■ Stopwords (removed)</span> 
           <span style="background-color: #ccffcc; padding: 5px; margin-left: 10px;"><b>■ Content Words (kept)</b></span></p>
        <div style="margin-top: 15px;">
            {' '.join(highlighted)}
        </div>
    </div>
    """
    display(HTML(html))

highlight_stopwords(sample_text, max_words=150)

In [ ]:
# Top 20 words before and after
freq_with = Counter(tokens).most_common(20)
freq_without = Counter(tokens_no_stop).most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before
words, counts = zip(*freq_with)
axes[0].barh(range(len(words)), counts, color='teal')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].invert_yaxis()
axes[0].set_title('Top 20 Words WITH Stopwords', fontsize=14)
axes[0].set_xlabel('Frequency')

# After
words, counts = zip(*freq_without)
axes[1].barh(range(len(words)), counts, color='gold')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].invert_yaxis()
axes[1].set_title('Top 20 Words WITHOUT Stopwords', fontsize=14)
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
with open(sample_file, 'r', encoding='utf-8') as f:
    sample_text_full = f.read()# [:2000]  # First 2000 characters

In [ ]:
from wordcloud import WordCloud

# Get a sample text
tokens = word_tokenize(sample_text_full.lower())
tokens_no_stop = [t for t in tokens if t not in stop_words and t.isalpha()]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before stopword removal - DISABLE automatic stopword removal
wordcloud_with = WordCloud(
    width=800, 
    height=400, 
    background_color='white',
    stopwords=set(),  # Empty set - don't remove any words automatically!
    colormap='viridis',        # <-- colour map for frequency
).generate(' '.join(tokens))

axes[0].imshow(wordcloud_with, interpolation='bilinear')
axes[0].set_title('WITH Stopwords\n', fontsize=16)
axes[0].axis('off')

# After stopword removal - use our cleaned tokens
wordcloud_without = WordCloud(
    width=800, 
    height=400, 
    background_color='white',
    stopwords=set(),  # Also disable here since we already filtered tokens
    colormap='plasma',         # <-- different colour map for contrast
).generate(' '.join(tokens_no_stop))

axes[1].imshow(wordcloud_without, interpolation='bilinear')
axes[1].set_title('WITHOUT Stopwords\n', fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Removed {len(tokens) - len(tokens_no_stop)} stopwords ({(1 - len(tokens_no_stop)/len(tokens))*100:.1f}% of tokens)")

# **Critical thinking**

_What is the advantage and disadvantage of word cloud visualisation?_

Word clouds are visually appealing and widely used in presentations, but they have significant limitations for rigorous analysis. Consider these questions:

#### **Precision & Measurement**
1. **Quantitative information:** Can you tell the exact frequency of a word from a word cloud? What information is lost compared to a bar chart or table?

2. **Size perception:** Are humans good at judging relative sizes of words? Try this: in the word cloud above, is "reason" twice as frequent as "nature"? Can you tell without counting?

3. **Ranking ambiguity:** Can you easily identify the 5th or 10th most frequent word? How does this compare to a ranked list?

#### **Context & Relationships**
4. **Loss of context:** Word clouds show isolated words. Why might "natural law" be more meaningful than seeing "natural" and "law" separately? What context is lost?

5. **Temporal patterns:** Could you use word clouds to show how vocabulary changes over time? What would be a better visualization?

6. **Relationships between concepts:** Do word clouds show which words appear together? How would you visualize co-occurrence patterns instead?

#### **Visual Bias**
7. **Layout effects:** The position and orientation of words in a word cloud is often random. Does placement in the center feel more important than placement at the edges, even when frequencies are the same?

8. **Font and color:** How do design choices (color schemes, fonts) influence your interpretation? Could these create misleading impressions?

#### **Scientific Communication**
9. **Reproducibility:** If another researcher generated a word cloud from the same data, would it look identical? Why or why not? (Hint: layout algorithms often use randomization)

10. **Citation in papers:** Have you seen word clouds in peer-reviewed philosophy journals? Why might they be more common in blog posts and presentations than in academic publications?

#### **When (if ever) are word clouds appropriate?**
11. **Exploratory vs. confirmatory:** When might a word cloud be useful in *initial* data exploration? When should you switch to more rigorous methods?

12. **Audience matters:** Word clouds are popular in journalism and public communication. Why? What trade-off are we making between accessibility and precision?

#### **Better alternatives:**
What visualizations could replace word clouds for:
- Showing exact frequencies? (→ ?)
- Comparing vocabularies across texts? (→ ?)
- Tracking terms over time? (→ ?)
- Showing word relationships? (→ ?)

---

**Key Takeaway:** Word clouds are useful for *quick impressions* and *public communication*, but rarely appropriate for rigorous scholarly analysis. Always ask: "What am I sacrificing in precision and detail for visual appeal?"

 =============================================== YOUR THOUGHTS HERE ===============================================



---

# Part 7: Integration with Metadata

In [ ]:
# Save complete metadata with text statistics
output_file = OUTPUT_DIR / 'tables' / 'nb02-corpus-complete-metadata.csv'
df_complete.to_csv(output_file, index=False)

print(f"✓ Saved complete metadata: {output_file}")
print(f"  Columns: {len(df_complete.columns)}")
print(f"  Rows: {len(df_complete)}")
print(f"\nColumns included:")
for col in df_complete.columns:
    print(f"  - {col}")

# Part 8: Final Summary Report

In [ ]:
# Generate comprehensive summary report
summary_report = f"""
{'=' * 80}
PHILOSOPHY CORPUS ANALYSIS REPORT
{'=' * 80}
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

CORPUS COMPOSITION
{'-' * 80}
Total texts selected: {len(df_metadata)}
Successfully downloaded: {len(df_complete[df_complete['word_count'].notna()])}
Failed/missing: {len(df_complete[df_complete['word_count'].isna()])}

SIZE STATISTICS
{'-' * 80}
Total words: {df_complete['word_count'].sum():,.0f}
Total characters: {df_complete['char_count'].sum():,.0f}
Corpus size: {df_complete['char_count'].sum() / (1024 * 1024):.2f} MB

Average words per text: {df_complete['word_count'].mean():,.0f}
Median words per text: {df_complete['word_count'].median():,.0f}
Shortest text: {df_complete['word_count'].min():,.0f} words
Longest text: {df_complete['word_count'].max():,.0f} words

VOCABULARY
{'-' * 80}
Total unique words (corpus-wide): {df_complete['unique_words'].sum():,}
Average unique words per text: {df_complete['unique_words'].mean():,.0f}
Average type-token ratio: {df_complete['type_token_ratio'].mean():.3f}

print(f"\nTEMPORAL COVERAGE (using publication years)")
print('-' * 80)
print(f"Texts with temporal data: {len(df_dated)} ({len(df_dated)/len(df_complete)*100:.1f}%)")
print(f"Date range: {df_dated['publication_year'].min():.0f} - {df_dated['publication_year'].max():.0f}")

Texts by period:
"""

# Add century breakdown
for century, count in century_counts.items():
    century_name = f"{int((century-1)*100)}s" if century > 0 else f"{int((century-1)*100)}s BCE"
    pct = count / len(df_dated) * 100
    summary_report += f"  {century_name:15} : {count:3} texts ({pct:5.1f}%)\n"

summary_report += f"""
AUTHOR REPRESENTATION
{'-' * 80}
Total unique authors: {df_complete['primary_author'].nunique()}

Top 10 authors by text count:
"""

for author, count in author_counts.head(10).items():
    summary_report += f"  {author:40} : {count} texts\n"

summary_report += f"""
SUBJECT DISTRIBUTION
{'-' * 80}
Total unique subjects: {len(subject_counts)}

Top 10 subjects:
"""

for subject, count in subject_counts.most_common(10):
    summary_report += f"  {count:3} | {subject}\n"

summary_report += f"""
DATA QUALITY NOTES
{'-' * 80}
Very short texts (< 10,000 words): {len(short_texts)}
Very long texts (> 300,000 words): {len(long_texts)}
Missing/failed texts: {len(missing_texts)}

OUTPUTS GENERATED
{'-' * 80}
- nb02-corpus-complete-metadata.csv (integrated metadata + text stats)
- text_quality_check_0_STUDENT_NAME.csv (for raw vs cleaned analysis)
- text_quality_check_1_STUDENT_NAME.csv (for text quality inspection)
- Visualisations: nb02-text_length_distribution.png, 
                  nb02-vocabulary_richness.png,
                  nb02-temporal_distribution.png,
                  nb02-author_representation.png,
                  nb02-subject_distribution.png

{'=' * 80}
END OF REPORT
{'=' * 80}
"""

# Save report
report_file = OUTPUT_DIR / 'reports' / 'nb02-corpus_analysis_report.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(summary_report)

print(summary_report)
print(f"\n✓ Report saved: {report_file}")

In [ ]:
# List all generated outputs
print("\n" + "=" * 80)
print("ALL GENERATED OUTPUTS")
print("=" * 80)

output_files = sorted(OUTPUT_DIR.glob('*'))
for f in output_files:
    size = f.stat().st_size / 1024  # KB
    print(f"  {f.name:50} ({size:8.1f} KB)")

print("\n✓ Corpus analysis complete!")

In [ ]:
def show_tree(path: Path, max_depth=4, depth=0):
    if depth > max_depth:
        return
    entries = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    for p in entries:
        if p.name.startswith(".") or p.name in {"__pycache__", ".ipynb_checkpoints"}:
            continue
        prefix = "  " * depth + ("- " if depth else "")
        if p.is_dir():
            print(f"{prefix}{p.name}/")
            show_tree(p, max_depth=max_depth, depth=depth+1)
        else:
            print(f"{prefix}{p.name}")

In [ ]:
show_tree(OUTPUT_DIR, max_depth=2)

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A2 highlight;
```